In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- citation_models_fillna ---

# --- citation_models_rank ---

print("✅ Fixtures loaded")
df = pd.DataFrame({
    "publication_date_rank": [0.5,0.3,0.8,0.1],
    "citation_count_rank": [0.2,0.7,0.4,0.9],
    "publication_date": pd.to_datetime(["2021-01-01","2022-06-15","2020-03-20","2023-09-01"]),
    "citationcount_document": [100,50,200,25],
    "citationcount_author": [500,300,800,100],
})
df_pl = pl.from_pandas(df)


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_citation_models_fillna():
    return df.assign(publication_date_rank=df["publication_date_rank"].fillna(len(df)))
    return None

def before_citation_models_rank():
    return df.assign(
        publication_date_rank=df["publication_date"].rank(ascending=False),
        citationcount_document_rank=df["citationcount_document"].rank(ascending=False),
        citationcount_author_rank=df["citationcount_author"].rank(ascending=False),
    )
    return None

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_citation_models_fillna():
    return df_pl.with_columns(pl.col("publication_date_rank").fill_null(len(df)).alias("publication_date_rank"))
    return None

def gen_citation_models_rank():
    return df_pl.with_columns(
        publication_date_rank=pl.col("publication_date").rank(method="average", descending=True),
        citationcount_document_rank=pl.col("citationcount_document").rank(method="average", descending=True),
        citationcount_author_rank=pl.col("citationcount_author").rank(method="average", descending=True),
    )
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: citation_models_fillna ===

_BASE_CITATION_DF = pd.DataFrame({
    "publication_date_rank": [0.5,0.3,0.8,0.1],
    "citation_count_rank": [0.2,0.7,0.4,0.9],
    "publication_date": pd.to_datetime(["2021-01-01","2022-06-15","2020-03-20","2023-09-01"]),
    "citationcount_document": [100,50,200,25],
    "citationcount_author": [500,300,800,100],
})

def _set_citation_before(frame_pd=None):
    global df, df_pl
    df = (frame_pd.copy() if frame_pd is not None else _BASE_CITATION_DF.copy())
    df_pl = pl.from_pandas(df)

def _set_citation_generated(frame_pd=None):
    global df, df_pl
    _pd = frame_pd.copy() if frame_pd is not None else _BASE_CITATION_DF.copy()
    df = pl.from_pandas(_pd)
    df_pl = df

# L1 smoke – generated
try:
    _set_citation_generated()
    _r = gen_citation_models_fillna()
    print("✅ L1 smoke gen_citation_models_fillna: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_citation_models_fillna: {type(_e).__name__}: {_e}")
finally:
    _set_citation_before()

# L1 smoke – before
try:
    _set_citation_before()
    _rb = before_citation_models_fillna()
    print("✅ L1 smoke before_citation_models_fillna: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_citation_models_fillna: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _set_citation_before()
    _rb = before_citation_models_fillna()
    _set_citation_generated()
    _rg = gen_citation_models_fillna()
    compare(_rb, _rg, "citation_models_fillna")
except Exception as _e:
    print(f"❌ L2 equivalence citation_models_fillna: setup error — {type(_e).__name__}: {_e}")
finally:
    _set_citation_before()

# L3 — null publication_date_rank values are replaced with len(df).
try:
    edge = _BASE_CITATION_DF.copy()
    edge.loc[[0, 2], "publication_date_rank"] = np.nan
    _set_citation_before(edge)
    _rb = before_citation_models_fillna()
    _set_citation_generated(edge)
    _rg = gen_citation_models_fillna()
    assert _rg.get_column("publication_date_rank").to_list()[0] == len(edge)
    compare(_rb, _rg, "L3 citation_models_fillna partial nulls")
except Exception as _e:
    print(f"❌ L3 citation_models_fillna partial nulls: {type(_e).__name__}: {_e}")
finally:
    _set_citation_before()

# L3 — all ranks null become len(df).
try:
    edge = _BASE_CITATION_DF.copy()
    edge["publication_date_rank"] = np.nan
    _set_citation_before(edge)
    _rb = before_citation_models_fillna()
    _set_citation_generated(edge)
    _rg = gen_citation_models_fillna()
    assert _rg.get_column("publication_date_rank").to_list() == [len(edge)] * len(edge)
    compare(_rb, _rg, "L3 citation_models_fillna all nulls")
except Exception as _e:
    print(f"❌ L3 citation_models_fillna all nulls: {type(_e).__name__}: {_e}")
finally:
    _set_citation_before()

# L3 — empty frame keeps schema and does not crash.
try:
    edge = _BASE_CITATION_DF.head(0)
    _set_citation_before(edge)
    _rb = before_citation_models_fillna()
    _set_citation_generated(edge)
    _rg = gen_citation_models_fillna()
    assert _rg.height == 0
    compare(_rb, _rg, "L3 citation_models_fillna empty frame")
except Exception as _e:
    print(f"❌ L3 citation_models_fillna empty frame: {type(_e).__name__}: {_e}")
finally:
    _set_citation_before()